# Extractor

This will create a pickle file which contains the info needed for the learner and theory evaluator.

I don't think I need to add prolog/clingo clauses... that happens in the `amati_bea.py` program.

In [1]:
import pandas as pd

from pyrsistent import pmap

from itertools import count
from string import whitespace
import pickle

from compound_split import char_split, doc_split
import phunspell

## Part 1: load in the data

In [2]:
with open(
    "//Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/2way/ALICE_LP_train_2way__v2.json"
) as fIn:
    j1_df = pd.read_json(fIn)

j1_df["partition"] = "train"

j1_df.head()

,id,question_id,question,answer,rubric,score,partition
0,73b0b485-5e89-40d8-ae68-4a650d4670c4,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,Bei einem höheren Zerteilungsgrad ( hier das Z...,{'Incorrect': ['Die Schüler:innen beurteilen d...,Incorrect,train
1,0c46f560-fcb4-4ea4-a540-cc2b96512b1c,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Das Größte apfelstück , also das mildem gering...",{'Incorrect': ['Die Schüler:innen beurteilen d...,Correct,train
2,467514d6-45d6-4f33-a1aa-dab58cadb9a7,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,verderben schneller\n,{'Incorrect': ['Die Schüler:innen beurteilen d...,Incorrect,train
3,b570cf0d-d29d-4add-bc9a-53156bb68003,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,Durch zerteilen der Äpfel wird dessen oberfläc...,{'Incorrect': ['Die Schüler:innen beurteilen d...,Correct,train
4,64d40458-107c-403e-bc30-f51ff3b09495,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je größer die Oberfläche vom Apfel ist , desto...",{'Incorrect': ['Die Schüler:innen beurteilen d...,Incorrect,train


In [3]:
with open(
    "/Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/2way/ALICE_LP_trial_2way___v2.json"
) as fIn:
    j2_df = pd.read_json(fIn)

j2_df["partition"] = "trial"

j2_df.head()

,id,question_id,question,answer,rubric,score,partition
0,f069275a-e786-4ba5-8c15-4349c138c15e,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,Durch Zerteilen der Äpfel ändert sich das Verh...,{'Incorrect': ['Die Schüler:innen beurteilen d...,Incorrect,trial
1,659094ae-3233-4a20-9da7-200b3eed9738,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,Zerteilter Apfel = höherer Zerteilungsgrad = h...,{'Incorrect': ['Die Schüler:innen beurteilen d...,Correct,trial
2,52fb2775-9aed-46c4-8388-d4bb4076cc7e,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je öfter der Apfel zerteilt wurde , desto höhe...",{'Incorrect': ['Die Schüler:innen beurteilen d...,Correct,trial
3,ad9c1e35-0ac1-4b17-9032-b052d77da325,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,äpfel verderben schneller bei höheren temperat...,{'Incorrect': ['Die Schüler:innen beurteilen d...,Incorrect,trial
4,b8ce1bd1-3d2b-4c0b-8da3-96534539b9cc,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je kleiner die Äpfel geschnitten werden , dest...",{'Incorrect': ['Die Schüler:innen beurteilen d...,Correct,trial


In [4]:
j_df = pd.concat([j1_df, j2_df], ignore_index=True)

j_df

,id,question_id,question,answer,rubric,score,partition
0,73b0b485-5e89-40d8-ae68-4a650d4670c4,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,Bei einem höheren Zerteilungsgrad ( hier das Z...,{'Incorrect': ['Die Schüler:innen beurteilen d...,Incorrect,train
1,0c46f560-fcb4-4ea4-a540-cc2b96512b1c,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Das Größte apfelstück , also das mildem gering...",{'Incorrect': ['Die Schüler:innen beurteilen d...,Correct,train
2,467514d6-45d6-4f33-a1aa-dab58cadb9a7,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,verderben schneller\n,{'Incorrect': ['Die Schüler:innen beurteilen d...,Incorrect,train
3,b570cf0d-d29d-4add-bc9a-53156bb68003,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,Durch zerteilen der Äpfel wird dessen oberfläc...,{'Incorrect': ['Die Schüler:innen beurteilen d...,Correct,train
4,64d40458-107c-403e-bc30-f51ff3b09495,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je größer die Oberfläche vom Apfel ist , desto...",{'Incorrect': ['Die Schüler:innen beurteilen d...,Incorrect,train
...,...,...,...,...,...,...,...
7894,d9702059-4d24-4e5e-bb24-7a66cbb57547,e71649c1-57e4-4cf6-a5fe-c55b8568add3,Beschreibe Auffälligkeiten der dargestellten d...,Nicht bei jeder Kollision sind die Edukte zu e...,{'Incorrect': ['Die Schüler:innen beschreiben ...,Correct,trial
7895,240f6b33-a928-4604-9b5f-03b25e40c432,e71649c1-57e4-4cf6-a5fe-c55b8568add3,Beschreibe Auffälligkeiten der dargestellten d...,Nicht alle Kollisionen der Edukt-Teilchen führ...,{'Incorrect': ['Die Schüler:innen beschreiben ...,Correct,trial
7896,874dd924-a661-4cad-a6bb-8649c0f2b51f,e71649c1-57e4-4cf6-a5fe-c55b8568add3,Beschreibe Auffälligkeiten der dargestellten d...,Hab ich doch in Aufgabe 2 gemacht\n,{'Incorrect': ['Die Schüler:innen beschreiben ...,Incorrect,trial
7897,af90be9a-a1fd-4cd2-9ac3-45ff308d429a,e71649c1-57e4-4cf6-a5fe-c55b8568add3,Beschreibe Auffälligkeiten der dargestellten d...,Nicht immer führt eine Kollision zu einer Reak...,{'Incorrect': ['Die Schüler:innen beschreiben ...,Incorrect,trial


In [5]:
j_df["rubric"].apply(tuple).drop_duplicates()

0    (Incorrect, Correct)
Name: rubric, dtype: object

There's a `Correct` and an `Incorrect` rubric for each question. There are two versions of each of the Incorrect rubrics.

Let's tidy things up a bit so the ids are more readable.

In [6]:
question_ids = {
    q_id: f"q_{str(q1).zfill(2)}"
    for (q_id, q1) in zip(set(j_df["question_id"]), count())
}

question_ids

{'fcf5ff56-5e98-4eb7-8df3-afb0544d9739': 'q_00',
 'c51b89a6-0219-4eee-bb89-b2b5168998f5': 'q_01',
 '379ba896-bb8f-440d-9d32-5dc9ca0297fe': 'q_02',
 'cb99dfb3-2788-4a10-b934-7ea8276b2b44': 'q_03',
 'b85b1a95-35ba-473e-a164-4602a4a13a2f': 'q_04',
 '0a8e8837-0a58-4e89-96c2-c44ae7281535': 'q_05',
 '509987fb-fce6-4d2f-98b8-bb4ba3761aae': 'q_06',
 'dad43e50-3167-41d6-996c-5b6e5f200c07': 'q_07',
 '278f349f-1552-41c4-bec2-1de41838f229': 'q_08',
 'bbb0e987-436a-46f3-b16d-34fd708f8236': 'q_09',
 '945f0e58-761c-4c89-af96-0261821fcf11': 'q_10',
 'e5e2e91d-9054-4996-a3bf-62518e06a07f': 'q_11',
 '52f6e329-eea6-423b-8d6a-047448d02c3f': 'q_12',
 '11be53c7-c020-445b-b3a4-409d4ba16347': 'q_13',
 'c8ee61ed-aa7c-44da-a43d-f862fa5cce8d': 'q_14',
 '48071573-9e44-4ae9-b08c-2b19b6ee58d6': 'q_15',
 '1f2f0773-54bb-4b72-963e-c3c946a1f53f': 'q_16',
 '18fab27c-7420-4b80-a5b4-cce94a9c315c': 'q_17',
 '1ed16c88-ff9d-488c-be13-6ba2be1d1cc0': 'q_18',
 'dee4136e-e922-4b83-8033-24350ea1610c': 'q_19',
 '06908246-ff47-414c

In [7]:
response_ids = {
    r_id: f"r_{str(r1).zfill(4)}" for (r_id, r1) in zip(set(j_df["id"]), count())
}

response_ids

{'654462e4-1609-45cc-990c-aaff6e4e0d6b': 'r_0000',
 '5574aa4f-3e40-4939-aa1f-d8382f073330': 'r_0001',
 'a46214c9-49b1-4574-8c17-55c3d9443526': 'r_0002',
 '4f92a6aa-7d89-4223-b9e4-0e2a062c78c0': 'r_0003',
 'af4a4c5c-1c20-4e58-9092-1f9c39a3e620': 'r_0004',
 '574fb264-2f58-46c7-bff0-7b9413071f4a': 'r_0005',
 '04a4fee4-1dab-48d2-903b-a4159319d652': 'r_0006',
 '292b4ea4-4656-4a2b-9251-74b59c49f191': 'r_0007',
 '5a3751d4-8e92-461c-893d-2122852113d2': 'r_0008',
 '138bd7c9-9340-4ac6-99ec-4e5ed2ac80c5': 'r_0009',
 'fff59cf2-6900-4ef0-99c6-c40b8c1414b9': 'r_0010',
 'f846b367-a4ec-4f75-9530-10154d94e28b': 'r_0011',
 '31cde4be-b2b7-4d8d-8517-729d3267158b': 'r_0012',
 'd49e5db1-8757-4718-8b51-7ec94f328fc5': 'r_0013',
 'b3c04cc4-1af0-4275-91cc-55435df17851': 'r_0014',
 '6a4064d4-a436-4071-a2d1-ce74eab6ed6e': 'r_0015',
 'c16bd155-16b8-4919-8a59-6921130dfa59': 'r_0016',
 '8f24c48c-b28a-4fc9-9631-1f73557d31dc': 'r_0017',
 '61eb9c35-592a-40a9-aa6b-05f53b1469a5': 'r_0018',
 '112ebdb0-b1de-4ffb-9444-892a7

In [10]:
id_mapper = pmap(question_ids).update(response_ids)
id_mapper['084bbcf2-f51e-4b15-8c6e-d648a2b2cc73']

'r_0996'

I'm going to convert the score to lower case as it'll make everything easier in clingo.

In [16]:
# Note to students: don't write code like this!

bea_2way_data_df = (
    j_df.rename(
        {
            "id": "original_response_id",
            "question_id": "original_question_id",
            "answer": "response",
            "score": "capitalised_score",
        },
        axis="columns",
    )
    .assign(
        response_id=j_df["id"].map(id_mapper),
        question_id=j_df["question_id"].map(id_mapper),
        correct=j_df["rubric"].apply(lambda x: x["Correct"]),
        incorrect_1=j_df["rubric"].apply(lambda x: x["Incorrect"][0]),
        incorrect_2=j_df["rubric"].apply(lambda x: x["Incorrect"][1]),
        
        
        score=j_df["score"]
        .str.lower()
        
         )
    .drop(["rubric", "capitalised_score"], axis="columns")
)

bea_2way_data_df.head()

,original_response_id,original_question_id,question,response,partition,response_id,question_id,correct,incorrect_1,incorrect_2,score
0,73b0b485-5e89-40d8-ae68-4a650d4670c4,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,Bei einem höheren Zerteilungsgrad ( hier das Z...,train,r_7726,q_71,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,incorrect
1,0c46f560-fcb4-4ea4-a540-cc2b96512b1c,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Das Größte apfelstück , also das mildem gering...",train,r_0724,q_71,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,correct
2,467514d6-45d6-4f33-a1aa-dab58cadb9a7,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,verderben schneller\n,train,r_3209,q_71,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,incorrect
3,b570cf0d-d29d-4add-bc9a-53156bb68003,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,Durch zerteilen der Äpfel wird dessen oberfläc...,train,r_0102,q_71,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,correct
4,64d40458-107c-403e-bc30-f51ff3b09495,920df344-fe5c-4f48-ae64-a8458e8db40a,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je größer die Oberfläche vom Apfel ist , desto...",train,r_6205,q_71,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,incorrect


In [31]:
questions_df = bea_2way_data_df[
    [
        "original_question_id",
        "question_id",
        "question",
        "correct",
        "incorrect_1",
        "incorrect_2",
    ]
]

questions_df = questions_df.assign(
    correct_id=questions_df["question_id"].replace(r"q(.*)", r"c\1", regex=True),
    incorrect_1_id=questions_df["question_id"].replace(
        r"q(.*)", r"i\1", regex=True
    ),
    incorrect_2_id=questions_df["question_id"].replace(r"q(.*)", r"j\1", regex=True),
)

questions_df = questions_df.drop_duplicates().sort_values(
    "question_id", ignore_index=True
)
questions_df.head()

,original_question_id,question_id,question,correct,incorrect_1,incorrect_2,correct_id,incorrect_1_id,incorrect_2_id
0,fcf5ff56-5e98-4eb7-8df3-afb0544d9739,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,"Die SuS erklären, dass durch die unterschiedli...","Die SuS erklären nicht, dass dadurch die Mögli...","Die SuS erklären, dass durch die unterschiedli...",c_00,i_00,j_00
1,c51b89a6-0219-4eee-bb89-b2b5168998f5,q_01,Begründe deine Auswahl.,Die Schüler:innen stellen einen Zusammenhang z...,Die Schüler:innen stellen keinen Zusammenhang ...,Die Schüler:innen stellen einen Zusammenhang z...,c_01,i_01,j_01
2,379ba896-bb8f-440d-9d32-5dc9ca0297fe,q_02,Formuliere selbst eine Fragestellung.,Die SuS beschreiben die durchschnittliche Gesc...,Die SuS beschreiben weder die durchschnittlich...,Die SuS beschreiben entweder die durchschnittl...,c_02,i_02,j_02
3,cb99dfb3-2788-4a10-b934-7ea8276b2b44,q_03,Beschreibe auf Basis Deines Vorwissens die Erk...,"Die Schüler:innen erläutern den Vorteil, dass ...",Die Schüler:innen erläutern den erweiterten Bl...,"Die Schüler:innen erläuten den Vorteil, dass d...",c_03,i_03,j_03
4,b85b1a95-35ba-473e-a164-4602a4a13a2f,q_04,In dem Video siehst du die abstrakte Funktions...,Die SuS beschreiben die Funktionsweise eines D...,Die SuS beschreiben die Funktionsweise nicht k...,"Die SuS beschreiben nur einen Teil, entweder d...",c_04,i_04,j_04


In [32]:
responses_df = bea_2way_data_df[
    [
        "original_response_id",
        "response_id",
        "question_id",
        "response",
        "score",
        "partition",
    ]
].sort_values(["question_id", "partition", "response_id"], ignore_index=True)


responses_df.head()

,original_response_id,response_id,question_id,response,score,partition
0,e78122dd-9f3c-456e-a546-fc905b023807,r_0111,q_00,Im Winter ist weniger Strahlung weil die Sonne...,incorrect,train
1,4cf98df5-8d79-4144-aeea-37df3bec5de5,r_0115,q_00,Es kann im Süden und Sommer mehr elektrische E...,incorrect,train
2,6550b62c-cc65-458d-a609-a7a38c45cbca,r_0129,q_00,da im sommer mehr die somme scheint gibt es au...,incorrect,train
3,e4d9c6e1-ed55-468a-9101-89218a989852,r_0178,q_00,im Sommer gewinnt man die meiste Energie wobei...,incorrect,train
4,7d3e3263-668f-4283-ab26-edd6d4c43be1,r_0292,q_00,Im Winter ist am wenigsten Energie da im Winte...,incorrect,train


## Part 2: Extract lemmata with SpaCy

We will use SpaCy to convert the sentences into a usable form.

In [33]:
import spacy
import contextualSpellCheck

In [34]:
nlp = spacy.load("de_core_news_md")

# The spell check is causing issues at the moment. I'll return to this later...
# contextualSpellCheck.add_to_pipe(nlp)

In [35]:
de = nlp("junge reicht auch langsam mal")
de

junge reicht auch langsam mal

Don't need to be particularly finnicky with this in the first instance. We have essentially four types of sentence: question, correct answer, incorrect "answer", and response. We've indexed them differently, so we can just (?) bung them all into a big DataFrame:

In [36]:
# Again, stern note to students: don't write code like this!!

text_df = pd.concat(
    [
        questions_df[["question_id", "question"]].rename(
            {"question_id": "id", "question": "text"}, axis="columns"
        ),
        questions_df[["correct_id", "correct"]].rename(
            {"correct": "text", "correct_id": "id"}, axis="columns"
        ),
        questions_df[["incorrect_1_id", "incorrect_1"]].rename(
            {"incorrect_1": "text", "incorrect_1_id": "id"}, axis="columns"
        ),
        questions_df[["incorrect_2_id", "incorrect_2"]].rename(
            {"incorrect_2": "text", "incorrect_2_id": "id"}, axis="columns"
        ),
        responses_df[["response_id", "response"]].rename(
            {"response_id": "id", "response": "text"}, axis="columns"
        ),
    ],
    ignore_index=True,
)

text_df

,id,text
0,q_00,Erläutere die Auswirkungen der jahreszeitabhän...
1,q_01,Begründe deine Auswahl.
2,q_02,Formuliere selbst eine Fragestellung.
3,q_03,Beschreibe auf Basis Deines Vorwissens die Erk...
4,q_04,In dem Video siehst du die abstrakte Funktions...
...,...,...
8206,r_7866,er wird größer\n
8207,r_1396,"Nein , weil sich gar nichts in der Spanne von ..."
8208,r_4071,Weil dadurch es zwei identische x und y Werte ...
8209,r_6273,"Weil ,\n"


OK, so now we can apply SpaCy to each of the texts to get the appropriate data:

In [37]:
text_df = text_df.assign(spacy=text_df["text"].apply(nlp))
text_df

,id,text,spacy
0,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,"(Erläutere, die, Auswirkungen, der, jahreszeit..."
1,q_01,Begründe deine Auswahl.,"(Begründe, deine, Auswahl, .)"
2,q_02,Formuliere selbst eine Fragestellung.,"(Formuliere, selbst, eine, Fragestellung, .)"
3,q_03,Beschreibe auf Basis Deines Vorwissens die Erk...,"(Beschreibe, auf, Basis, Deines, Vorwissens, d..."
4,q_04,In dem Video siehst du die abstrakte Funktions...,"(In, dem, Video, siehst, du, die, abstrakte, F..."
...,...,...,...
8206,r_7866,er wird größer\n,"(er, wird, größer, \n)"
8207,r_1396,"Nein , weil sich gar nichts in der Spanne von ...","(Nein, ,, weil, sich, gar, nichts, in, der, Sp..."
8208,r_4071,Weil dadurch es zwei identische x und y Werte ...,"(Weil, dadurch, es, zwei, identische, x, und, ..."
8209,r_6273,"Weil ,\n","(Weil, ,, \n)"


I'm putting all the lemmata into lower case... I suspect this is naughty, but I'm going to see how it goes first.

In [38]:
text_df = text_df.explode("spacy", ignore_index=True)
text_df = text_df.assign(
    i=text_df["spacy"].apply(lambda x: x.i),
    lemma=text_df["spacy"].apply(lambda x: x.lemma_.lower()),
    word_text=text_df["spacy"].apply(lambda x: x.text),
)
text_df

,id,text,spacy,i,lemma,word_text
0,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,Erläutere,0,erläuter,Erläutere
1,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,die,1,der,die
2,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,Auswirkungen,2,auswirkung,Auswirkungen
3,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,der,3,der,der
4,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,jahreszeitabhängigen,4,jahreszeitabhängig,jahreszeitabhängigen
...,...,...,...,...,...,...
275450,r_6877,Da der Differenzquotient die Steigerung zwisch...,genau,18,genau,genau
275451,r_6877,Da der Differenzquotient die Steigerung zwisch...,aufeinander,19,aufeinander,aufeinander
275452,r_6877,Da der Differenzquotient die Steigerung zwisch...,liegen,20,liegen,liegen
275453,r_6877,Da der Differenzquotient die Steigerung zwisch...,.,21,--,.


I think I'm also going to remove any cases where the lemma is just whitespace.

In [39]:
def whitespace_str(str_in):
    return all([c in whitespace for c in str_in])


assert whitespace_str(
    """    
                      
                                """
)
assert not whitespace_str("slkdjf  s")

In [40]:
text_df = text_df[-text_df["lemma"].apply(whitespace_str)]

-----------------------------

In [41]:
questions_df

,original_question_id,question_id,question,correct,incorrect_1,incorrect_2,correct_id,incorrect_1_id,incorrect_2_id
0,fcf5ff56-5e98-4eb7-8df3-afb0544d9739,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,"Die SuS erklären, dass durch die unterschiedli...","Die SuS erklären nicht, dass dadurch die Mögli...","Die SuS erklären, dass durch die unterschiedli...",c_00,i_00,j_00
1,c51b89a6-0219-4eee-bb89-b2b5168998f5,q_01,Begründe deine Auswahl.,Die Schüler:innen stellen einen Zusammenhang z...,Die Schüler:innen stellen keinen Zusammenhang ...,Die Schüler:innen stellen einen Zusammenhang z...,c_01,i_01,j_01
2,379ba896-bb8f-440d-9d32-5dc9ca0297fe,q_02,Formuliere selbst eine Fragestellung.,Die SuS beschreiben die durchschnittliche Gesc...,Die SuS beschreiben weder die durchschnittlich...,Die SuS beschreiben entweder die durchschnittl...,c_02,i_02,j_02
3,cb99dfb3-2788-4a10-b934-7ea8276b2b44,q_03,Beschreibe auf Basis Deines Vorwissens die Erk...,"Die Schüler:innen erläutern den Vorteil, dass ...",Die Schüler:innen erläutern den erweiterten Bl...,"Die Schüler:innen erläuten den Vorteil, dass d...",c_03,i_03,j_03
4,b85b1a95-35ba-473e-a164-4602a4a13a2f,q_04,In dem Video siehst du die abstrakte Funktions...,Die SuS beschreiben die Funktionsweise eines D...,Die SuS beschreiben die Funktionsweise nicht k...,"Die SuS beschreiben nur einen Teil, entweder d...",c_04,i_04,j_04
...,...,...,...,...,...,...,...,...,...
73,afdd3dad-858c-46e7-972e-d5909efbe9ce,q_73,Vergleiche mit Hinblick auf die Temperatur die...,Die MB-Verteilung wird aufgabenbezogen korrekt...,Die MB-Verteilung wird weder aufgabenbezogen k...,Die MB-Verteilung wird aufgabenbezogen korrekt...,c_73,i_73,j_73
74,35d897f0-c1b4-422c-9a33-43e7da1ffe27,q_74,Schau dir deine Hypothesen noch einmal an. Ent...,Die Erklärung ...\n... beschreibt das Vorhande...,Die Erklärung \n... beschreibt das Vorhandense...,Die Erklärung ...\n... beschreibt das Vorhande...,c_74,i_74,j_74
75,b470d591-dc0c-41e1-afa7-4f212d6248e5,q_75,Beschreibe die geschwindigkeitsbezogene Häufig...,Die MB-Verteilung wird aufgabenbezogen korrekt...,Die MB-Verteilung wird weder aufgabenbezogen k...,Die MB-Verteilung wird aufgabenbezogen korrekt...,c_75,i_75,j_75
76,e71649c1-57e4-4cf6-a5fe-c55b8568add3,q_76,Beschreibe Auffälligkeiten der dargestellten d...,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargstellte ...,c_76,i_76,j_76


In [42]:
responses_df

,original_response_id,response_id,question_id,response,score,partition
0,e78122dd-9f3c-456e-a546-fc905b023807,r_0111,q_00,Im Winter ist weniger Strahlung weil die Sonne...,incorrect,train
1,4cf98df5-8d79-4144-aeea-37df3bec5de5,r_0115,q_00,Es kann im Süden und Sommer mehr elektrische E...,incorrect,train
2,6550b62c-cc65-458d-a609-a7a38c45cbca,r_0129,q_00,da im sommer mehr die somme scheint gibt es au...,incorrect,train
3,e4d9c6e1-ed55-468a-9101-89218a989852,r_0178,q_00,im Sommer gewinnt man die meiste Energie wobei...,incorrect,train
4,7d3e3263-668f-4283-ab26-edd6d4c43be1,r_0292,q_00,Im Winter ist am wenigsten Energie da im Winte...,incorrect,train
...,...,...,...,...,...,...
7894,1bd6d708-9b88-4033-a387-455a2b20fa07,r_7866,q_77,er wird größer\n,incorrect,train
7895,1e434c36-86f1-4338-be09-25d5a18479cf,r_1396,q_77,"Nein , weil sich gar nichts in der Spanne von ...",incorrect,trial
7896,541bfa0a-71f8-4ec3-8760-c3b2e495144d,r_4071,q_77,Weil dadurch es zwei identische x und y Werte ...,incorrect,trial
7897,4dd48587-e1f6-4ffb-87a7-16965d4cca82,r_6273,q_77,"Weil ,\n",incorrect,trial


Quickly drop the `spacy` column, as that isn't picklable.

In [43]:
text_df.drop("spacy", axis="columns")

,id,text,i,lemma,word_text
0,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,0,erläuter,Erläutere
1,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,1,der,die
2,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,2,auswirkung,Auswirkungen
3,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,3,der,der
4,q_00,Erläutere die Auswirkungen der jahreszeitabhän...,4,jahreszeitabhängig,jahreszeitabhängigen
...,...,...,...,...,...
275449,r_6877,Da der Differenzquotient die Steigerung zwisch...,17,punkt,Punkte
275450,r_6877,Da der Differenzquotient die Steigerung zwisch...,18,genau,genau
275451,r_6877,Da der Differenzquotient die Steigerung zwisch...,19,aufeinander,aufeinander
275452,r_6877,Da der Differenzquotient die Steigerung zwisch...,20,liegen,liegen


There's one other wrinkle: some of the lemmata contain quotation marks, which screws up the training file. As an interim hacky solution, I'll simply swap any instances of `"` for `'`. I think that the `"` appear in code, so it's not a sensible switch, but it'll do for the moment.

In [44]:
text_df = text_df.drop("lemma", axis="columns").assign(
    lemma=text_df["lemma"].str.replace('"', "'")
)

text_df.query(
    """
              `word_text`.str.contains('"')"""
).sort_values("word_text")

,id,text,spacy,i,word_text,lemma
81,q_05,Öffne das Material für deine Gruppe! Schaue di...,"""",18,"""",--
160432,r_4999,"( a ) Durch den Begriff "" Gleichgewicht "" hat ...","""",8,"""",--
160462,r_4999,"( a ) Durch den Begriff "" Gleichgewicht "" hat ...","""",38,"""",--
160464,r_4999,"( a ) Durch den Begriff "" Gleichgewicht "" hat ...","""",40,"""",--
179531,r_4166,Vegleichende Versuchsbeobachtungen Ansatz A An...,"""",75,"""",--
...,...,...,...,...,...,...
253306,r_2527,Beantwortung der Impulsfrage Mögliche Darstell...,"id=""MathJax-Element-1-Frame",18,"id=""MathJax-Element-1-Frame",id='mathjax-element-1-fram
252459,r_0919,Beantwortung der Impulsfrage Mögliche Darstell...,"role=""presentation",28,"role=""presentation",role='presentation
253312,r_2527,Beantwortung der Impulsfrage Mögliche Darstell...,"role=""presentation",24,"role=""presentation",role='presentation
253308,r_2527,Beantwortung der Impulsfrage Mögliche Darstell...,"tabindex=""0",20,"tabindex=""0",tabindex='0


OK, and I think that those are in the format that I can export them.

In [45]:
with open("bea_2way_data.pickle", "wb") as fOut:
    pickle.dump(
        {
            "questions": questions_df,
            "responses": responses_df,
            "text": text_df.drop("spacy", axis="columns"),
        },
        fOut,
    )